# Exploring DuckDB & SQLMesh

Exploring FHIR data infrastructure with DuckDB and SQLMesh

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 02/03/2026 (SGT)   | Martin | Create   | Notebook created. DuckDB exploration | 
| 17/03/2026 (SGT)   | Martin | Update   | Exploration of post-pipeline tables | 
| 23/03/2026 (SGT)   | Martin | Update   | Flattening data from other tables into silver | 

# Content

* [DuckDB](#duckdb)

# DuckDB

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()

False

## Merging files

For systems that cannot handle docker files, use this to create a single merged DB

In [2]:
base_path = "../data/warehouse"
source_files = os.listdir(base_path)
source_files = [f"{base_path}/{file}" for file in source_files]
print(source_files)
target_db_path = "../data/merged_database.duckdb"

con = duckdb.connect(target_db_path)
con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

for i, file_path in enumerate(source_files):
  con.execute(f"ATTACH '{file_path}' AS source{i}")

  tables = con.execute(f"SELECT table_name FROM information_schema.tables WHERE table_catalog = 'source{i}'").fetchall()
  print(tables)
  for table_row in tables:
    table_name = table_row[0]
    table_name = f"bronze.{table_name}"

    con.execute(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM source{i}.{table_name} LIMIT 0")
    con.execute(f"INSERT INTO {table_name} SELECT * FROM source{i}.{table_name}")

for i, file_path in enumerate(source_files):
  con.execute(f"DETACH source{i}")

con.close()

print(f"Data from {source_files} has been merged into {target_db_path}")

['../data/warehouse/mimic_fhir.duckdb', '../data/warehouse/mimic_fhir_condition.duckdb', '../data/warehouse/mimic_fhir_disp_org_obs.duckdb', '../data/warehouse/mimic_fhir_encountered.duckdb', '../data/warehouse/mimic_fhir_location.duckdb', '../data/warehouse/mimic_fhir_obs.duckdb', '../data/warehouse/mimic_fhir_patient.duckdb']
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]
[('fhir_resources',)]
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data from ['../data/warehouse/mimic_fhir.duckdb', '../data/warehouse/mimic_fhir_condition.duckdb', '../data/warehouse/mimic_fhir_disp_org_obs.duckdb', '../data/warehouse/mimic_fhir_encountered.duckdb', '../data/warehouse/mimic_fhir_location.duckdb', '../data/warehouse/mimic_fhir_obs.duckdb', '../data/warehouse/mimic_fhir_patient.duckdb'] has been merged into ../data/merged_database.duckdb


In [ ]:
con = duckdb.connect(source_files[1])
tables = con.sql("SHOW TABLES FROM bronze").df()
print(tables)
con.close()

# Exploring Tables

Bronze tables:

- `bronze.fhir_resources`: Collection of raw FHIR resource data
- `bronze.synthea`: Synthea generated data

Silver tables:

- `silver.obs_vital`: Patient vitals recorded in long format

Intermediate tables:

- `intermediate.vitals_wide`: Flattened vitals table containing patient recorded vitals (e.g heart rate, temperature, ...)

Gold (marts) tables:

- `marts.vitals_eda`: 

In [78]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")
dbs = con.sql("SELECT * FROM duckdb_databases").df()
dbs

,database_name,database_oid,path,comment,tags,internal,type,readonly,encrypted,cipher
0,mimic_fhir,727,/Users/martz/Repos/MIMIC-FHIR/notebooks/../dat...,None,{'storage_version': 'v1.0.0+'},False,duckdb,False,False,None


In [79]:
tables = con.sql("SHOW ALL TABLES").df()
tables

,database,schema,name,column_names,column_types,temporary
0,mimic_fhir,bronze,fhir_resources,"[resource_type, resource_id, resource, source_...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR]",False
1,mimic_fhir,bronze,synthea,"[full_url, resource_type, resource, request, s...","[VARCHAR, VARCHAR, JSON, JSON, VARCHAR, VARCHAR]",False
2,mimic_fhir,intermediate,vitals_wide,"[encounter_id, patient_id, effective_datetime,...","[VARCHAR, VARCHAR, TIMESTAMP, DOUBLE, DOUBLE, ...",False
3,mimic_fhir,marts,vitals_eda,"[encounter_id, patient_id, effective_datetime,...","[VARCHAR, VARCHAR, TIMESTAMP, DOUBLE, DOUBLE, ...",False
4,mimic_fhir,silver,condition,"[resource_id, icd_code, icd_name, patient_id, ...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
5,mimic_fhir,silver,encounter,"[resource_id, class_code, class_name, start_ti...","[VARCHAR, VARCHAR, VARCHAR, TIMESTAMP, TIMESTA...",False
6,mimic_fhir,silver,location,"[resource_id, name, is_active, loaction_physic...","[VARCHAR, VARCHAR, INTEGER, VARCHAR, VARCHAR]",False
7,mimic_fhir,silver,medication_dispense,"[resource_id, status, encounter_id, patient_id...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
8,mimic_fhir,silver,obs_vitals,"[resource_id, patient_id, encounter_id, effect...","[VARCHAR, VARCHAR, VARCHAR, TIMESTAMP, VARCHAR...",False
9,mimic_fhir,silver,organization,"[resource_id, name, is_active, npi_value]","[VARCHAR, VARCHAR, VARCHAR, INTEGER]",False


In [9]:
con.sql("SELECT * FROM bronze.synthea LIMIT 10")

┌───────────────────────────────────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## Exploring Bronze tables

Looking to see if other sqlmesh pipelines can be created from the data

In [7]:
con.sql("SELECT * FROM bronze.fhir_resources LIMIT 10;")

┌───────────────┬──────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────┐
│ resource_type │             resource_id              │                                                                                                                                                                                                                                                                            resource                                             

In [8]:
con.sql("SELECT DISTINCT resource_type FROM bronze.fhir_resources;")

┌────────────────────┐
│   resource_type    │
│      varchar       │
├────────────────────┤
│ Organization       │
│ Location           │
│ Observation        │
│ Procedure          │
│ Condition          │
│ MedicationDispense │
│ Encounter          │
│ Patient            │
└────────────────────┘

### 1. Organization

[Source](https://build.fhir.org/organization.html)

Contains a single organisation: "Beth Israel Deaconess Medical Center"

In [80]:
# Silver table flatten query
con.sql("""
  SELECT 
    resource->>'id' AS resource_id,
    resource->>'name' AS name,
    resource->>'active' AS is_active,
    TRY_CAST(resource->'identifier'->0->>'value' AS INT32) AS npi_value
  FROM bronze.fhir_resources
  WHERE resource_type='Organization';
""")

┌──────────────────────────────────────┬──────────────────────────────────────┬───────────┬────────────┐
│             resource_id              │                 name                 │ is_active │ npi_value  │
│               varchar                │               varchar                │  varchar  │   int32    │
├──────────────────────────────────────┼──────────────────────────────────────┼───────────┼────────────┤
│ ee172322-118b-5716-abbc-18e4c5437e15 │ Beth Israel Deaconess Medical Center │ true      │ 1194052720 │
└──────────────────────────────────────┴──────────────────────────────────────┴───────────┴────────────┘

In [81]:
# Silver table test query
con.sql("SELECT * FROM silver.organization")

┌──────────────────────────────────────┬──────────────────────────────────────┬───────────┬────────────┐
│             resource_id              │                 name                 │ is_active │ npi_value  │
│               varchar                │               varchar                │  varchar  │   int32    │
├──────────────────────────────────────┼──────────────────────────────────────┼───────────┼────────────┤
│ ee172322-118b-5716-abbc-18e4c5437e15 │ Beth Israel Deaconess Medical Center │ true      │ 1194052720 │
└──────────────────────────────────────┴──────────────────────────────────────┴───────────┴────────────┘

### 2. Location

[Source](https://build.fhir.org/location.html)

39 total locations. Each is a specialist clinic in the hospital

In [ ]:
# Flatten Location resource
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->>'name' AS name,
    CASE WHEN resource->>'status' = 'active' THEN 1 ELSE 0 END AS is_active,
    resource->'physicalType'->'coding'->0->>'display' AS loaction_physical_type,
    regexp_replace(resource->'managingOrganization'->>'reference', 'Organization/', '') AS organization_id
  FROM bronze.fhir_resources
  WHERE resource_type='Location'
  LIMIT 10;
""")

┌──────────────────────────────────────┬──────────────────────────────────────────────────┬───────────┬────────────────────────┬──────────────────────────────────────┐
│             resource_id              │                       name                       │ is_active │ loaction_physical_type │           organization_id            │
│               varchar                │                     varchar                      │   int32   │        varchar         │               varchar                │
├──────────────────────────────────────┼──────────────────────────────────────────────────┼───────────┼────────────────────────┼──────────────────────────────────────┤
│ ecbf468a-22ec-5320-8e11-6ebcc918dad5 │ Cardiology Surgery Intermediate                  │         1 │ Ward                   │ ee172322-118b-5716-abbc-18e4c5437e15 │
│ 501cd59a-cd8a-5f98-8298-2ca9c897d59f │ Emergency Department                             │         1 │ Ward                   │ ee172322-118b-5716-abbc-18e4c54

In [82]:
# Check silver.location table
con.sql("SELECT * FROM silver.location LIMIT 10;")

┌──────────────────────────────────────┬──────────────────────────────────────────────────┬───────────┬────────────────────────┬──────────────────────────────────────┐
│             resource_id              │                       name                       │ is_active │ loaction_physical_type │           organization_id            │
│               varchar                │                     varchar                      │   int32   │        varchar         │               varchar                │
├──────────────────────────────────────┼──────────────────────────────────────────────────┼───────────┼────────────────────────┼──────────────────────────────────────┤
│ ecbf468a-22ec-5320-8e11-6ebcc918dad5 │ Cardiology Surgery Intermediate                  │         1 │ Ward                   │ ee172322-118b-5716-abbc-18e4c5437e15 │
│ 501cd59a-cd8a-5f98-8298-2ca9c897d59f │ Emergency Department                             │         1 │ Ward                   │ ee172322-118b-5716-abbc-18e4c54

### 3. Procedure

- [Source](https://build.fhir.org/location.html)
- [Procedure Codes SNOMED_CT](https://build.fhir.org/valueset-procedure-code.html)

1,989,697 total procedure entries. Each has `patient_id` and an `encounter_id`

In [44]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->'code'->'coding'->0->>'code' AS snomed_ct_id,
    regexp_replace(resource->'code'->'coding'->0->>'display', ' \(procedure\)', '') AS snomed_ct_procedure,
    resource->>'status' AS status,
    regexp_replace(resource->'subject'->>'reference', 'Patient/', '') AS patient_id,
    regexp_replace(resource->'encounter'->>'reference', 'Encounter/', '') AS encounter_id,
    (resource->>'performedDateTime')::TIMESTAMP AS performed_datetime
  FROM bronze.fhir_resources
  WHERE resource_type='Procedure'
  LIMIT 10;
""")

┌──────────────────────────────────────┬──────────────┬───────────────────────────────────────┬───────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┐
│             resource_id              │ snomed_ct_id │          snomed_ct_procedure          │  status   │              patient_id              │             encounter_id             │ performed_datetime  │
│               varchar                │   varchar    │                varchar                │  varchar  │               varchar                │               varchar                │      timestamp      │
├──────────────────────────────────────┼──────────────┼───────────────────────────────────────┼───────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┤
│ ef1cc99a-f637-5480-b646-0526caa35b16 │ 410188000    │ Taking patient vital signs assessment │ completed │ 000000ba-735e-5858-a92d-b856d73dd69a │ 9142a2a0-fb3e-5aee-96

In [83]:
# Check silver.procedure
con.sql("SELECT * FROM silver.procedure LIMIT 10;")

┌──────────────────────────────────────┬──────────────┬───────────────────────────────────────┬───────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┐
│             resource_id              │ snomed_ct_id │          snomed_ct_procedure          │  status   │              patient_id              │             encounter_id             │ performed_datetime  │
│               varchar                │   varchar    │                varchar                │  varchar  │               varchar                │               varchar                │      timestamp      │
├──────────────────────────────────────┼──────────────┼───────────────────────────────────────┼───────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┤
│ ef1cc99a-f637-5480-b646-0526caa35b16 │ 410188000    │ Taking patient vital signs assessment │ completed │ 000000ba-735e-5858-a92d-b856d73dd69a │ 9142a2a0-fb3e-5aee-96

### 4. Condition

[Source](https://build.fhir.org/condition.html)

899,050 entries in condition table.

Column Details:
- `condtion_code`: [ICD-10 coding](https://www.icd10data.com/)
- `condtion_category`: "problem-list-item" | "encounter-diagnosis" [Source](https://build.fhir.org/valueset-condition-category.html)

In [50]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->'code'->'coding'->0->>'code' AS icd_code,
    resource->'code'->'coding'->0->>'display' AS icd_name,
    regexp_replace(resource->'subject'->>'reference', 'Patient/', '') AS patient_id,
    resource->'category'->0->'coding'->0->>'code' AS condition_category,
    regexp_replace(resource->'encounter'->>'reference', 'Encounter/', '') AS encounter_id
  FROM bronze.fhir_resources
  WHERE resource_type='Condition'
  LIMIT 10;
""")

┌──────────────────────────────────────┬──────────┬─────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┐
│             resource_id              │ icd_code │                                icd_name                                 │              patient_id              │ condition_category  │             encounter_id             │
│               varchar                │ varchar  │                                 varchar                                 │               varchar                │       varchar       │               varchar                │
├──────────────────────────────────────┼──────────┼─────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┤
│ 3ec3187d-b760-531a-a02d-a69b58b66fe1 │ T402X5A  │ Adverse effect of other opioids, initial enc

In [84]:
# Test silver.condition
con.sql("SELECT * FROM silver.condition LIMIT 10;")

┌──────────────────────────────────────┬──────────┬─────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┐
│             resource_id              │ icd_code │                            icd_name                             │              patient_id              │ condition_category  │             encounter_id             │
│               varchar                │ varchar  │                             varchar                             │               varchar                │       varchar       │               varchar                │
├──────────────────────────────────────┼──────────┼─────────────────────────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┤
│ c9647885-10fa-5b71-83ce-e8f471ffadf3 │ R079     │ Chest pain, unspecified                                         │ 23062449-6

### 5. Medication Dispense

[Source](https://build.fhir.org/medicationdispense.html)

1,586,053 entries in bronze

In [60]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->>'status' AS status,
    regexp_replace(resource->'context'->>'reference', 'Encounter/', '') AS encounter_id,
    regexp_replace(resource->'subject'->>'reference', 'Patient/', '') AS patient_id,
    resource->'medicationCodeableConcept'->'coding'->0->>'code' AS medication_code,
    resource->'medicationCodeableConcept'->>'text' AS medication_name,
    (resource->>'whenHandedOver')::TIMESTAMP AS handed_over_timestamp
  FROM bronze.fhir_resources
  WHERE resource_type='MedicationDispense'
  LIMIT 10;
""")

┌──────────────────────────────────────┬─────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────┬──────────────────────────────┬───────────────────────┐
│             resource_id              │ status  │             encounter_id             │              patient_id              │ medication_code │       medication_name        │ handed_over_timestamp │
│               varchar                │ varchar │               varchar                │               varchar                │     varchar     │           varchar            │       timestamp       │
├──────────────────────────────────────┼─────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────┼──────────────────────────────┼───────────────────────┤
│ a98720b0-042b-5084-8d3a-2f326e89f08a │ unknown │ 9142a2a0-fb3e-5aee-96b8-d0fc3fc605f9 │ 000000ba-735e-5858-a92d-b856d73dd69a │ 078733          │ LORazepam 2mg/1mL 1mL SYR    │ 2169-10-24 21:

In [85]:
# Test silver.medication_dispense
con.sql("SELECT * FROM silver.medication_dispense LIMIT 10;")

┌──────────────────────────────────────┬─────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────┬─────────────────────────────┬───────────────────────┐
│             resource_id              │ status  │             encounter_id             │              patient_id              │ medication_code │       medication_name       │ handed_over_timestamp │
│               varchar                │ varchar │               varchar                │               varchar                │     varchar     │           varchar           │       timestamp       │
├──────────────────────────────────────┼─────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────┼─────────────────────────────┼───────────────────────┤
│ 9f3fcc49-d9f1-5efc-9ba2-2c620bfce473 │ unknown │ e2bf0c31-f54c-5280-9174-c76554cb2ab8 │ 13dc4151-7e7c-5c32-9a60-cf2683d3a668 │ 038861          │ Lidocaine Jelly 2% (Urojet) │ 2171-02-10 11:40:00

## 6. Encounter

[Source](https://build.fhir.org/encounter.html)

425,087 entries in encounter


In [70]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->'class'->>'code' AS class_code,
    resource->'class'->>'display' AS class_name,
    (resource->'period'->>'start')::TIMESTAMP AS start_timestamp,
    (resource->'period'->>'end')::TIMESTAMP AS end_timestamp,
    resource->>'status' AS status,
    regexp_replace(resource->'subject'->>'reference', 'Patient/', '') AS patient_id,
    resource->'hospitalization'->'admitSource'->'coding'->0->>'code' AS admit_source,
    resource->'hospitalization'->'dischargeDisposition'->'coding'->0->>'code' AS discharge_disposition,
    regexp_replace(resource->'serviceProvider'->>'reference', 'Organization/', '') AS organization_id
  FROM bronze.fhir_resources
  WHERE resource_type='Encounter'
  LIMIT 10;
""")

┌──────────────────────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────┬──────────┬──────────────────────────────────────┬──────────────┬───────────────────────┬──────────────────────────────────────┐
│             resource_id              │ class_code │ class_name │   start_timestamp   │    end_timestamp    │  status  │              patient_id              │ admit_source │ discharge_disposition │           organization_id            │
│               varchar                │  varchar   │  varchar   │      timestamp      │      timestamp      │ varchar  │               varchar                │   varchar    │        varchar        │               varchar                │
├──────────────────────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────┼──────────┼──────────────────────────────────────┼──────────────┼───────────────────────┼──────────────────────────────────────┤
│ ab67e413-be5f-5c55-97d2-615c1941d950 │ EME

In [86]:
# Text silver.encounter
con.sql("SELECT * FROM silver.encounter LIMIT 10;")

┌──────────────────────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────┬──────────┬──────────────────────────────────────┬──────────────┬───────────────────────┬──────────────────────────────────────┐
│             resource_id              │ class_code │ class_name │   start_timestamp   │    end_timestamp    │  status  │              patient_id              │ admit_source │ discharge_disposition │           organization_id            │
│               varchar                │  varchar   │  varchar   │      timestamp      │      timestamp      │ varchar  │               varchar                │   varchar    │        varchar        │               varchar                │
├──────────────────────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────┼──────────┼──────────────────────────────────────┼──────────────┼───────────────────────┼──────────────────────────────────────┤
│ ab67e413-be5f-5c55-97d2-615c1941d950 │ EME

### 7. Patient

[Source](https://build.fhir.org/patient.html)

299,712 patients in data

In [74]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->'name'->0->>'family' AS name,
    resource->>'gender' AS gender,
    resource->>'birthDate' AS birth_date,
    resource->'extension'->0->'extension'->0->'valueCoding'->>'display' AS race,
    resource->'extension'->1->'extension'->0->'valueCoding'->>'display' AS ethnicity,
    resource->'identifier'->0->>'value' AS patient_num,
    resource->'communication'->0->'language'->'coding'->0->>'code' AS language,
    resource->'maritalStatus'->'coding'->0->>'code' AS marital_status,
    regexp_replace(resource->'managingOrganization'->>'reference', 'Organization/', '') AS organization_id
  FROM bronze.fhir_resources
  WHERE resource_type='Patient'
  LIMIT 10;
""")

┌──────────────────────────────────────┬──────────────────┬─────────┬────────────┬─────────┬────────────────────────┬─────────────┬──────────┬────────────────┬──────────────────────────────────────┐
│             resource_id              │       name       │ gender  │ birth_date │  race   │       ethnicity        │ patient_num │ language │ marital_status │           organization_id            │
│               varchar                │     varchar      │ varchar │  varchar   │ varchar │        varchar         │   varchar   │ varchar  │    varchar     │               varchar                │
├──────────────────────────────────────┼──────────────────┼─────────┼────────────┼─────────┼────────────────────────┼─────────────┼──────────┼────────────────┼──────────────────────────────────────┤
│ 0a8eebfd-a352-522e-89f0-1d4a13abdebc │ Patient_10000032 │ female  │ 2128-05-06 │ White   │ Not Hispanic or Latino │ 10000032    │ en       │ W              │ ee172322-118b-5716-abbc-18e4c5437e15 │
│ 55b

In [87]:
# Test silver.patient
con.sql("SELECT * FROM silver.patient LIMIT 10;")

┌──────────────────────────────────────┬──────────────────┬─────────┬────────────┬─────────┬────────────────────────┬─────────────┬──────────┬────────────────┬──────────────────────────────────────┐
│             resource_id              │       name       │ gender  │ birth_date │  race   │       ethnicity        │ patient_num │ language │ marital_status │           organization_id            │
│               varchar                │     varchar      │ varchar │  varchar   │ varchar │        varchar         │   varchar   │ varchar  │    varchar     │               varchar                │
├──────────────────────────────────────┼──────────────────┼─────────┼────────────┼─────────┼────────────────────────┼─────────────┼──────────┼────────────────┼──────────────────────────────────────┤
│ 7d101f5f-aba3-586c-94b2-6055c90992a2 │ Patient_15173979 │ male    │ 2072-02-02 │ unknown │ NULL                   │ 15173979    │ en       │ S              │ ee172322-118b-5716-abbc-18e4c5437e15 │
│ a08

---

## Silver & Gold tables

Exploring Silver and Gold (mart) tables

In [20]:
# silver.obs_vital
con.sql("""
  SELECT *
  FROM silver.obs_vitals
  LIMIT 10
""")

┌──────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬────────┬────────────────┐
│             resource_id              │              patient_id              │             encounter_id             │ effective_datetime  │ loinc_code │ value  │      unit      │
│               varchar                │               varchar                │               varchar                │      timestamp      │  varchar   │ double │    varchar     │
├──────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼────────┼────────────────┤
│ e6944724-62ed-53df-b56e-f26d77f8fc14 │ 000000ba-735e-5858-a92d-b856d73dd69a │ ab67e413-be5f-5c55-97d2-615c1941d950 │ 2169-09-06 17:35:00 │ 9279-1     │   18.0 │ breaths/minute │
│ 0df7928c-9982-5ca9-ab43-b965e5ec275f │ 000000ba-735e-5858-a92d-b856d73dd69a │ ab67e413-be5f-5c55-9

In [ ]:
# intermediate.vitals_wide
con.sql("SELECT * FROM intermediate.vitals_wide LIMIT 10")

┌──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬──────────┬──────────┬────────────┬───────────┬───────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬─────────────┐
│             encounter_id             │              patient_id              │ effective_datetime  │ temp_value │ hr_value │ rr_value │ spo2_value │ sbp_value │ dbp_value │ temp_present │ hr_present │ rr_present │ spo2_present │ sbp_present │ dbp_present │
│               varchar                │               varchar                │      timestamp      │   double   │  double  │  double  │   double   │  double   │  double   │    int32     │   int32    │   int32    │    int32     │    int32    │    int32    │
├──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼──────────┼──────────┼────────────┼───────────┼───────────┼──────────────┼────────────┼────────────┼────────────

In [17]:
con.sql("""
  SELECT * 
  FROM intermediate.vitals_wide
  WHERE patient_id='fb348f58-e2e7-5fab-8118-494943e3b84b'
  ORDER BY effective_datetime DESC
  LIMIT 10
""")


┌──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬──────────┬──────────┬────────────┬───────────┬───────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬─────────────┐
│             encounter_id             │              patient_id              │ effective_datetime  │ temp_value │ hr_value │ rr_value │ spo2_value │ sbp_value │ dbp_value │ temp_present │ hr_present │ rr_present │ spo2_present │ sbp_present │ dbp_present │
│               varchar                │               varchar                │      timestamp      │   double   │  double  │  double  │   double   │  double   │  double   │    int32     │   int32    │   int32    │    int32     │    int32    │    int32    │
├──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼──────────┼──────────┼────────────┼───────────┼───────────┼──────────────┼────────────┼────────────┼────────────

In [3]:
# marts.vitals_eda
con.sql("""
  SELECT *
  FROM marts.vitals_eda
  LIMIT 10
""")

┌──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬──────────┬──────────┬────────────┬───────────┬───────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬─────────────┬────────────┬────────────┬────────────┬─────────────┬─────────────┬──────────────┬────────────────────────┬───────────┬────────────────────────┬─────────────────┐
│             encounter_id             │              patient_id              │ effective_datetime  │ temp_value │ hr_value │ rr_value │ spo2_value │ sbp_value │ dbp_value │ temp_present │ hr_present │ rr_present │ spo2_present │ sbp_present │ dbp_present │ n_vitals_6 │ n_vitals_5 │ n_vitals_3 │ hour_of_day │ day_of_week │ obs_position │ total_obs_in_encounter │ delta_min │ encounter_duration_hrs │ encounter_phase │
│               varchar                │               varchar                │      timestamp      │   double   │  double  │  double  │   double   │  double   

In [5]:
con.sql("SELECT * FROM bronze.fhir_resources LIMIT 10")

┌───────────────┬──────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────┐
│ resource_type │             resource_id              │                                                                                                                                                                                                                                                                            resource                                             

---

# Synthea Data

Flattening Synthea data into a workable format

In [1]:
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")

In [5]:
con.sql("SELECT DISTINCT resource_type FROM bronze.synthea;")

┌──────────────────────────┐
│      resource_type       │
│         varchar          │
├──────────────────────────┤
│ ExplanationOfBenefit     │
│ Observation              │
│ ImagingStudy             │
│ Encounter                │
│ DiagnosticReport         │
│ Immunization             │
│ CarePlan                 │
│ MedicationAdministration │
│ DocumentReference        │
│ Provenance               │
│ SupplyDelivery           │
│ Condition                │
│ Procedure                │
│ Medication               │
│ Device                   │
│ AllergyIntolerance       │
│ MedicationRequest        │
│ Patient                  │
│ Claim                    │
│ CareTeam                 │
├──────────────────────────┤
│         20 rows          │
└──────────────────────────┘

In [6]:
con.sql("SELECT * FROM bronze.synthea WHERE resource_type='Patient' LIMIT 10;")

┌───────────────────────────────────────────────┬───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## 1. Patient

In [16]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->'name'->0->'given'->>0 AS first_name,
    resource->'name'->0->'given'->>1 AS middle_name,
    resource->'name'->0->>'family' AS last_name,
    resource->>'gender' AS gender,
    resource->'extension'->0->'extension'->0->'valueCoding'->>'display' AS race,
    resource->'extension'->1->'extension'->1->>'valueString' AS ethnicity,
    resource->>'birthDate' AS birth_date,
    resource->'telecom'->0->>'value' AS phone_number,
    resource->'maritalStatus'->>'text' AS marital_status,
    resource->'address'->0->>'city' AS city,
    resource->'address'->0->>'state' AS state,
    resource->'address'->0->>'country' AS country,
    resource->'address'->0->'line'->>0 AS street_address,
    TRY_CAST(resource->'address'->0->'extension'->0->'extension'->0->'valueDecimal' AS DOUBLE) AS latitude,
    TRY_CAST(resource->'address'->0->'extension'->0->'extension'->1->'valueDecimal' AS DOUBLE) AS longitude,
    TRY_CAST(resource->'extension'->5->'valueDecimal' AS DOUBLE) AS disablility_adjusted_life_years,
    TRY_CAST(resource->'extension'->6->'valueDecimal' AS DOUBLE) AS quality_adjusted_life_years
  FROM bronze.synthea
  WHERE resource_type='Patient';
""")

┌──────────────────────────────────────┬─────────────┬─────────────┬────────────┬─────────┬───────────────────────────────────────────┬────────────────────────┬────────────┬──────────────┬────────────────┬──────────────────┬─────────┬─────────┬───────────────────────────────────────┬────────────────────┬────────────────────┬─────────────────────────────────┬─────────────────────────────┐
│             resource_id              │ first_name  │ middle_name │ last_name  │ gender  │                   race                    │       ethnicity        │ birth_date │ phone_number │ marital_status │       city       │  state  │ country │            street_address             │      latitude      │     longitude      │ disablility_adjusted_life_years │ quality_adjusted_life_years │
│               varchar                │   varchar   │   varchar   │  varchar   │ varchar │                  varchar                  │        varchar         │  varchar   │   varchar    │    varchar     │     varchar 

## 2. Observation

Information extracted from patients

In [24]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->>'status' AS status,
    resource->'category'->0->'coding'->0->>'code' AS record,
    resource->'code'->'coding'->0->>'code' AS measure_code,
    resource->'code'->'coding'->0->>'display' AS measure_name,
    TRY_CAST(
      CASE WHEN resource->'valueQuantity'->>'value' IS NOT NULL
      THEN resource->'valueQuantity'->>'value'
      ELSE resource->'component'->0->'valueQuantity'->>'value' END AS DOUBLE
    ) AS value,
    CASE WHEN resource->'valueQuantity'->>'unit' IS NOT NULL THEN resource->'valueQuantity'->>'unit' ELSE resource->'component'->0->'valueQuantity'->>'unit' END AS unit,
    regexp_replace(resource->'subject'->>'reference', 'urn:uuid:', '') AS patient_id,
    (resource->>'effectiveDateTime')::TIMESTAMP AS effective_timestamp,
    (resource->>'issued')::TIMESTAMP AS issued_timestamp
  FROM bronze.synthea
  WHERE resource_type='Observation'
  LIMIT 50;
""")

┌──────────────────────────────────────┬─────────┬────────────────┬──────────────┬──────────────────────────────────────────────────────────────────────────┬────────┬─────────┬──────────────────────────────────────┬─────────────────────┬─────────────────────────┐
│             resource_id              │ status  │     record     │ measure_code │                               measure_name                               │ value  │  unit   │              patient_id              │ effective_timestamp │    issued_timestamp     │
│               varchar                │ varchar │    varchar     │   varchar    │                                 varchar                                  │ double │ varchar │               varchar                │      timestamp      │        timestamp        │
├──────────────────────────────────────┼─────────┼────────────────┼──────────────┼──────────────────────────────────────────────────────────────────────────┼────────┼─────────┼────────────────────────────────

## 3. Encounter

In [31]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->>'status' AS status,
    resource->'class'->>'code' AS class,
    resource->'type'->0->'coding'->0->>'code' AS procedure_code,
    regexp_replace(resource->'type'->0->'coding'->0->>'display', ' \(procedure\)', '') AS procedure_name,
    regexp_replace(resource->'subject'->>'reference', 'urn:uuid:', '') AS patient_id,
    resource->'participant'->0->'type'->0->'coding'->0->>'code' AS practitioner_code,
    resource->'participant'->0->'type'->0->'coding'->0->>'display' AS practitioner_type,
    split_part(resource->'participant'->0->'individual'->>'reference', '|', 2) AS practitioner_id,
    resource->'participant'->0->'individual'->>'display' AS practitioner_name,
    (resource->'period'->>'start')::TIMESTAMP AS start_date,
    (resource->'period'->>'end')::TIMESTAMP AS end_date,
    split_part(resource->'location'->0->'location'->>'reference', '|', 2) AS location_id,
    resource->'location'->0->'location'->>'display' AS location_name,
    split_part(resource->'serviceProvider'->>'reference', '|', 2) AS provider_id,
    resource->'serviceProvider'->>'display' AS provider_name,
  FROM bronze.synthea
  WHERE resource_type='Encounter'
  LIMIT 10;
""")

┌──────────────────────────────────────┬──────────┬─────────┬────────────────┬────────────────────────────────┬──────────────────────────────────────┬───────────────────┬───────────────────┬─────────────────┬──────────────────────────────┬─────────────────────┬─────────────────────┬──────────────────────────────────────┬────────────────────────────────────┬──────────────────────────────────────┬────────────────────────────────────┐
│             resource_id              │  status  │  class  │ procedure_code │         procedure_name         │              patient_id              │ practitioner_code │ practitioner_type │ practitioner_id │      practitioner_name       │     start_date      │      end_date       │             location_id              │           location_name            │             provider_id              │           provider_name            │
│               varchar                │ varchar  │ varchar │    varchar     │            varchar             │               va

## 4. Condition

In [37]:
con.sql("""
  SELECT
    resource->>'id' AS resource_id,
    resource->'clinicalStatus'->'coding'->0->>'code' AS clinical_status,
    resource->'verificationStatus'->'coding'->0->>'code' AS verification_status,
    resource->'category'->0->'coding'->0->>'code' AS category,
    resource->'code'->'coding'->0->>'code' AS condition_code,
    regexp_replace(resource->'code'->'coding'->0->>'display', ' \((.*?)\)', '') AS condition,
    regexp_extract(resource->'code'->'coding'->0->>'display', '\((.*?)\)', 1) AS condition_type,
    regexp_replace(resource->'subject'->>'reference', 'urn:uuid:', '') AS patient_id,
    regexp_replace(resource->'encounter'->>'reference', 'urn:uuid:', '') AS encounter_id,
    (resource->>'onsetDateTime')::TIMESTAMP AS onset_timestamp,
    (resource->>'recordedDate')::TIMESTAMP AS recorded_timestamp,
  FROM bronze.synthea
  WHERE resource_type='Condition'
  LIMIT 10;
""")

┌──────────────────────────────────────┬─────────────────┬─────────────────────┬─────────────────────┬────────────────┬──────────────────────────────────────────┬────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬─────────────────────┐
│             resource_id              │ clinical_status │ verification_status │      category       │ condition_code │                condition                 │ condition_type │              patient_id              │             encounter_id             │   onset_timestamp   │ recorded_timestamp  │
│               varchar                │     varchar     │       varchar       │       varchar       │    varchar     │                 varchar                  │    varchar     │               varchar                │               varchar                │      timestamp      │      timestamp      │
├──────────────────────────────────────┼─────────────────┼─────────────────────┼──────────────

In [10]:
con.close()

In [ ]:
%load_ext watermark
%watermark